<div style="font-family:system-ui;border-left:6px solid #B8721A;padding:6px 16px">
<div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#566273">The AI-Ready Organisation · Lab 4 · Module 4 · ≈30 min</div>
<div style="font-size:26px;font-weight:800;margin:6px 0">See a data advantage form</div>
<div style="font-size:15px"><b>You will:</b> train the same model on growing slices of a marketplace's orders, find where extra data stops paying off, and judge how defensible a data advantage really is.</div>
<div style="font-size:13px;color:#566273;margin-top:6px">Dataset: Olist Brazilian E-Commerce (about 100,000 orders, public Kaggle dataset)</div>
</div>

**How to use this notebook** · No coding needed.
1. In the menu choose **Runtime → Run all** (in Kaggle: **Run All**). The first run takes 1–3 minutes.
2. Change values in the **form fields** on the right of each cell (sliders, dropdowns). The cell reruns when you change a value; if it does not, press the play button.
3. Read the plain-English findings under each chart and record your answers in the final cell.

If the real dataset cannot be downloaded (for example, without a Kaggle sign-in), the notebook automatically switches to a **realistic simulated copy with the same structure** and tells you so. The lessons still hold.

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import warnings, os, io, zipfile, urllib.request
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
plt.rcParams.update({"figure.figsize": (9, 4.4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 11, "axes.titleweight": "bold"})
BUILD, BUY, BLEND, INK, GOOD, BAD = "#B8721A", "#13807B", "#4255B0", "#111822", "#2D7A4C", "#B03A3A"
pd.set_option("display.max_columns", 30)

def say(text):
    display(Markdown(text))

def scorecard(title, rows):
    """rows: list of (label, value, note)"""
    body = "".join(f"<tr><td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px;font-weight:700;text-align:right'>{b}</td>"
                   f"<td style='padding:6px 12px;color:#566273'>{c}</td></tr>" for a, b, c in rows)
    display(HTML(f"<div style='font-family:system-ui;margin:8px 0'><div style='font-weight:700;font-size:15px;margin-bottom:4px'>{title}</div>"
                 f"<table style='border-collapse:collapse;font-size:14px'>{body}</table></div>"))

def try_kagglehub(handle, filename):
    """Download a public Kaggle dataset. Returns a file path or None."""
    try:
        try:
            import kagglehub
        except ImportError:
            os.system("pip -q install kagglehub")
            import kagglehub
        path = kagglehub.dataset_download(handle)
        for root, _, files in os.walk(path):
            if filename in files:
                return os.path.join(root, filename)
    except Exception as e:
        print(f"Kaggle download unavailable ({type(e).__name__}).")
    return None

RNG = np.random.default_rng(42)
DATA_NOTE = ""

## 1 · Build an order-level table
We join orders, items and reviews, then ask: **can we predict which orders will get a poor review (1 or 2 stars) before the customer writes it?**

In [ ]:
#@title Load and prepare data { display-mode: "form" }
files = ["olist_orders_dataset.csv", "olist_order_items_dataset.csv", "olist_order_reviews_dataset.csv", "olist_customers_dataset.csv", "olist_sellers_dataset.csv"]
p = try_kagglehub("olistbr/brazilian-ecommerce", files[0])
if p:
    base = os.path.dirname(p); o, it, rv, cu, se = [pd.read_csv(os.path.join(base, f)) for f in files]
    o = o[o.order_status == "delivered"].copy()
    for c in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]: o[c] = pd.to_datetime(o[c])
    it = it.merge(se[["seller_id", "seller_state"]], on="seller_id")
    agg = it.groupby("order_id").agg(price=("price", "sum"), freight=("freight_value", "sum"), n_items=("order_item_id", "count"),
                                     n_sellers=("seller_id", "nunique"), seller_state=("seller_state", "first"))
    d = o.merge(agg, on="order_id").merge(cu[["customer_id", "customer_state"]], on="customer_id").merge(rv.groupby("order_id")["review_score"].mean(), on="order_id")
    d["delay_days"] = (d.order_delivered_customer_date - d.order_estimated_delivery_date).dt.days
    d["delivery_days"] = (d.order_delivered_customer_date - d.order_purchase_timestamp).dt.days
    d["same_state"] = (d.seller_state == d.customer_state).astype(int); d["weekday"] = d.order_purchase_timestamp.dt.weekday
    d = d.dropna(subset=["delay_days", "delivery_days"])
    DATA_NOTE = "Real Olist marketplace data."
else:
    n = 96000
    delivery = np.clip(RNG.gamma(3, 4, n), 1, 90); est = delivery + RNG.normal(10, 7, n)
    price = RNG.gamma(2, 60, n); freight = price*RNG.uniform(.08, .3, n); n_items = RNG.choice([1, 1, 1, 2, 3, 4], n)
    n_sellers = np.minimum(n_items, RNG.choice([1, 1, 1, 2], n)); same = RNG.binomial(1, .35, n)
    delay = delivery - est
    logit = -1.7 + .17*np.clip(delay, -20, 40) + .9*(n_sellers > 1) + .002*freight - .3*same + RNG.normal(0, .9, n)
    d = pd.DataFrame({"price": price, "freight": freight, "n_items": n_items, "n_sellers": n_sellers, "same_state": same,
                      "delay_days": delay, "delivery_days": delivery, "weekday": RNG.integers(0, 7, n),
                      "review_score": np.where(RNG.random(n) < 1/(1+np.exp(-logit)), RNG.choice([1, 2], n), RNG.choice([3, 4, 5], n))})
    DATA_NOTE = "Simulated copy (Kaggle download unavailable). Same features as the Olist build."
d["poor_review"] = (d.review_score <= 2).astype(int)
FEATS = ["price", "freight", "n_items", "n_sellers", "same_state", "delay_days", "delivery_days", "weekday"]
say(f"**{DATA_NOTE}** {len(d):,} delivered orders; **{d.poor_review.mean():.0%}** received a poor review.")
display(d[FEATS + ["review_score"]].head())

## 2 · Train on 2% to 100% of the data
Each slice is trained three times and scored on the same held-out orders. The score is AUC: 0.5 is guessing, 1.0 is perfect.

In [ ]:
#@title Run the learning curve { display-mode: "form" }
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
tr, te = train_test_split(d, test_size=.25, random_state=3, stratify=d.poor_review)
fracs = [.02, .05, .1, .25, .5, 1.0]; rows = []
for f in fracs:
    for rep in range(3):
        s = tr.sample(frac=f, random_state=rep) if f < 1 else tr
        m = HistGradientBoostingClassifier(max_iter=150, random_state=rep).fit(s[FEATS], s.poor_review)
        rows.append((f, len(s), roc_auc_score(te.poor_review, m.predict_proba(te[FEATS])[:, 1])))
        if f == 1.0: break
lc = pd.DataFrame(rows, columns=["share", "orders", "auc"]).groupby("share").agg(orders=("orders", "mean"), auc=("auc", "mean")).reset_index()
lc["gain_vs_previous"] = lc.auc.diff()
flat = lc.loc[lc.gain_vs_previous < .005, "orders"]
flat_at = int(flat.iloc[0]) if len(flat) else None
fig, ax = plt.subplots()
ax.plot(lc.orders, lc.auc, marker="o", color=BLEND, lw=2.5); ax.set_xscale("log")
if flat_at: ax.axvline(flat_at, color=BUILD, ls="--"); ax.text(flat_at*1.05, lc.auc.min(), "curve flattens", color=BUILD)
ax.set(title="More data, better model? (log scale)", xlabel="Training orders", ylabel="AUC on held-out orders"); plt.show()
display(lc.round(3))
say(f"The model reaches AUC **{lc.auc.iloc[-1]:.2f}** with all the data. " + (f"Beyond roughly **{flat_at:,} orders**, doubling the data adds less than half a point." if flat_at else "The curve is still rising, so more data still helps."))

## 3 · How far behind is a smaller rival?
Set the share of your data a competitor holds.

In [ ]:
#@title Rival comparison { display-mode: "form" }
rival_share_pct = 25 #@param {type:"slider", min:2, max:100, step:1}
yours = lc.auc.iloc[-1]; rival = float(np.interp(np.log(rival_share_pct/100), np.log(lc.share), lc.auc))
poor_orders_year = 12000 #@param {type:"number"}
scorecard("Your model versus the rival's", [
    ("Your AUC (100% of data)", f"{yours:.3f}", ""),
    (f"Rival AUC ({rival_share_pct}% of data)", f"{rival:.3f}", "interpolated from the curve"),
    ("Gap", f"{(yours-rival)*100:.1f} pts", "AUC points")])
durable = (yours - rival) > .02
say(("### The data advantage is **meaningful** at this gap.\nA rival with this share of data is visibly worse, so the volume itself is part of the moat. Keep the loop running: more orders, more reviews, a better model." if durable else
     "### The data advantage is **thin** at this gap.\nA rival with this share of data performs almost as well. The edge must come from elsewhere: faster action on the prediction, better operations, or data rivals cannot get."))

## 4 · Place your own use case on the moat matrix
Go back to **Simulator 4.2** on the course page:
- **Hard to copy:** high only if rivals cannot obtain similar data or the process around it.
- **Feedback loop:** high only if every use of the model creates labelled data that improves it, as each review does here.

## Your reflection → capstone §4
Answer in the form fields, then run the cell. Copy the summary into §4 of your business case on the course page.

In [ ]:
#@title Record your answers { display-mode: "form" }
q1 = "" #@param {type:"string"}
q2 = "" #@param {type:"string"}
q3 = "" #@param {type:"string"}
answers = [("Where my use case sits on the moat matrix, and why:", q1), ("Does my data advantage flatten early or keep compounding?", q2), ("One action that would strengthen the feedback loop:", q3)]
say("### Lab 4 reflection")
for q, a in answers:
    say(f"**{q}**  \n{a if a.strip() else '_(not answered yet)_'}")